In [1]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_fact_loan_exposure
#
# Layer
# -----
# Gold Layer - Fact Tables
#
# Purpose
# -------
# Build the core loan-level exposure fact table used for
# credit portfolio analytics, ECL, stress testing, Power BI,
# semantic modelling and AI decision intelligence.
#
# Grain
# -----
# One row per loan.
#
# Output
# ------
# fact_loan_exposure
#
# Enterprise Concepts
# -------------------
# ✓ Fact Table
# ✓ Star Schema
# ✓ Dimensional Modelling
# ✓ Loan Exposure Analytics
# ✓ Basel III / RWA Foundation
# ✓ Semantic Model Foundation
# ============================================================

from pyspark.sql.functions import *
from datetime import datetime

source_loan_table = "silver_loan"
dim_customer_table = "dim_customer"
dim_country_table = "dim_country"
dim_industry_table = "dim_industry"

target_table = "fact_loan_exposure"
pipeline_name = "nb_build_fact_loan_exposure"

run_start_time = datetime.now()

print("ERIP Fact Loan Exposure Build Started")

StatementMeta(, 8d35974b-dc36-488e-b071-2dcac3b497f2, 3, Finished, Available, Finished, False)

ERIP Fact Loan Exposure Build Started


In [2]:
# ============================================================
# SECTION 2 - READ SOURCE TABLES
# ============================================================

silver_loan = spark.table(source_loan_table)
dim_customer = spark.table(dim_customer_table)
dim_country = spark.table(dim_country_table)
dim_industry = spark.table(dim_industry_table)

print(f"Silver Loan Rows : {silver_loan.count()}")
print(f"Dim Customer Rows: {dim_customer.count()}")
print(f"Dim Country Rows : {dim_country.count()}")
print(f"Dim Industry Rows: {dim_industry.count()}")

StatementMeta(, 8d35974b-dc36-488e-b071-2dcac3b497f2, 4, Finished, Available, Finished, False)

Silver Loan Rows : 5000
Dim Customer Rows: 1000
Dim Country Rows : 6
Dim Industry Rows: 10


In [3]:
# ============================================================
# SECTION 3 - BUILD FACT LOAN EXPOSURE
# ============================================================

fact_loan_exposure = (
    silver_loan.alias("l")
    .join(
        dim_customer
        .select(
            "customer_sk",
            "customer_id",
            "country_code",
            "industry_code"
        )
        .alias("c"),
        "customer_id",
        "left"
    )
    .join(
        dim_country
        .select("country_sk", "country_code")
        .alias("co"),
        "country_code",
        "left"
    )
    .join(
        dim_industry
        .select("industry_sk", "industry_code")
        .alias("i"),
        "industry_code",
        "left"
    )
    .select(
        col("l.loan_sk"),
        col("l.loan_id"),
        col("l.facility_id"),
        col("c.customer_sk"),
        col("co.country_sk"),
        col("i.industry_sk"),
        col("l.customer_id"),
        col("l.customer_group_id"),
        col("l.product_type"),
        col("l.facility_status"),
        col("l.origination_date"),
        col("l.maturity_date"),
        col("l.loan_tenure_years"),
        col("l.remaining_maturity_years"),
        col("l.approved_limit"),
        col("l.outstanding_balance"),
        col("l.undrawn_amount"),
        col("l.exposure_at_default"),
        col("l.utilization_pct"),
        col("l.exposure_band"),
        col("l.currency"),
        col("l.interest_rate_pct"),
        col("l.repayment_schedule"),
        col("l.secured_flag"),
        col("l.collateral_id"),
        col("l.collateral_type"),
        col("l.collateral_value"),
        col("l.loan_to_value_pct"),
        col("l.ifrs9_stage"),
        col("l.ifrs9_stage_numeric"),
        col("l.risk_weight"),
        col("l.risk_weighted_assets"),
        current_timestamp().alias("gold_updated_timestamp")
    )
)

print(f"Fact rows created: {fact_loan_exposure.count()}")
display(fact_loan_exposure.limit(10))

StatementMeta(, 8d35974b-dc36-488e-b071-2dcac3b497f2, 5, Finished, Available, Finished, False)

Fact rows created: 5000


SynapseWidget(Synapse.DataFrame, 1ea2da1e-4676-4ca6-8ee1-73d5dc75a8a6)

In [4]:
# ============================================================
# SECTION 4 - FACT TABLE QUALITY VALIDATION
# ============================================================

total_rows = fact_loan_exposure.count()

duplicate_loan_ids = total_rows - fact_loan_exposure.select("loan_id").distinct().count()

null_loan_sk = fact_loan_exposure.filter(col("loan_sk").isNull()).count()

null_customer_sk = fact_loan_exposure.filter(col("customer_sk").isNull()).count()

null_country_sk = fact_loan_exposure.filter(col("country_sk").isNull()).count()

null_industry_sk = fact_loan_exposure.filter(col("industry_sk").isNull()).count()

negative_ead = fact_loan_exposure.filter(col("exposure_at_default") < 0).count()

negative_rwa = fact_loan_exposure.filter(col("risk_weighted_assets") < 0).count()

print("Fact Loan Exposure Quality Checks")
print("--------------------------------")
print(f"Rows                 : {total_rows}")
print(f"Duplicate Loan IDs   : {duplicate_loan_ids}")
print(f"Null Loan SK         : {null_loan_sk}")
print(f"Null Customer SK     : {null_customer_sk}")
print(f"Null Country SK      : {null_country_sk}")
print(f"Null Industry SK     : {null_industry_sk}")
print(f"Negative EAD         : {negative_ead}")
print(f"Negative RWA         : {negative_rwa}")

if (
    duplicate_loan_ids > 0 or
    null_loan_sk > 0 or
    null_customer_sk > 0 or
    null_country_sk > 0 or
    null_industry_sk > 0 or
    negative_ead > 0 or
    negative_rwa > 0
):
    raise Exception("Fact Loan Exposure Validation Failed")
else:
    print("✓ Fact Loan Exposure Validation Passed")

StatementMeta(, 8d35974b-dc36-488e-b071-2dcac3b497f2, 6, Finished, Available, Finished, False)

Fact Loan Exposure Quality Checks
--------------------------------
Rows                 : 5000
Duplicate Loan IDs   : 0
Null Loan SK         : 0
Null Customer SK     : 0
Null Country SK      : 0
Null Industry SK     : 0
Negative EAD         : 0
Negative RWA         : 0
✓ Fact Loan Exposure Validation Passed


In [5]:
# ============================================================
# SECTION 5 - WRITE GOLD FACT TABLE
# ============================================================

(
    fact_loan_exposure.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(target_table)
)

print(f"✓ Gold fact table created: {target_table}")
print(f"Rows written: {fact_loan_exposure.count()}")

StatementMeta(, 8d35974b-dc36-488e-b071-2dcac3b497f2, 7, Finished, Available, Finished, False)

✓ Gold fact table created: fact_loan_exposure
Rows written: 5000
